In [ ]:
%reload_ext autoreload
%autoreload 2

# Agent-Based Task Execution

This notebook demonstrates how to use the `Agent` pipeline from **[OnPrem.LLM](https://github.com/amaiya/onprem)** to create autonomous agents that can execute complex tasks using a variety of tools.

The pipeline works with any LiteLLM-supported model that supports tool-calling:

- **Cloud**: openai/gpt-5.2-codex, anthropic/claude-sonnet-4-5, gemini/gemini-1.5-pro
- **Local**: Ollama (ollama/llama3.1), vLLM (hosted_vllm/<model>), llama.cpp (use OpenAI interface)

For **llama.cpp**:  Use `openai/<model_name>` (e.g., gpt-oss-120b) as model parameter and then set env variable `OPENAI_API_BASE=http://localhost:<port>/v1`

## The AgentExecutor

The `AgentExecutor` allows you to launch AI agents to solve various tasks using both cloud and local models. We will use **anthropic/claude-sonnet-4-5** (cloud) and **glm-4.7-flash** (local) for these examples.

By default, the `AgentExecutor` has access to 9 built-in tools.  You remove access to built-in-tools as necessary.  You can optionally give the agent access to **custom tools**, as we'll illustrate below.

The `AgentExecutor` is implemented using our coding agent, PatchPal, which you'll need to install: `pip install patchpal`.

In [ ]:
# | notest

from onprem.pipelines import AgentExecutor

In [ ]:
# | notest

AgentExecutor.print_default_tools()

AgentExecutor Default Tools

These tools are used by default when enabled_tools=None:

   1. read_file       - Read complete file contents
   2. read_lines      - Read specific line ranges from files
   3. edit_file       - Edit files via find/replace
   4. write_file      - Write complete file contents
   5. grep            - Search for patterns in files
   6. find            - Find files by glob pattern
   7. run_shell       - Execute shell commands
   8. web_search      - Search the web for information
   9. web_fetch       - Fetch and read content from URLs

Customization Examples:

# Use defaults (all tools including shell):
executor = AgentExecutor(model='anthropic/claude-sonnet-4-5')

# Defaults but no shell access (safer):
executor = AgentExecutor(
    model='openai/gpt-5-mini',
    disable_shell=True
)

# Minimal tools:
executor = AgentExecutor(
    model='openai/gpt-5-mini',
    enabled_tools=['read_file', 'write_file']
)

# Web research only:
executor = AgentExecutor(
    mo

## Examples

Let's run through some examples for different scenarios.

### Basic Calculator Example

In this introductory example, we will launch an agent to build a **calculator module in Python**.

By default, the agent will operate within the working_directory you specify (or the current folder if not working directory is specified).  The agent cannot read or write outside the working directory.

**Important Note:**:  If the agent has access to the `run_shell` tool, it can potentially read or modify files outside of the working directory (e.g., auto-generating and executing a Python script that writes files outside of working directory).  For these reasons, you can either supply the `disable_shell=True` to remove shell access or `sandbox=True`, which runs the agent in an ephemeral container.  

In this first example, we set `sandbox=True`. The example was run on Windows Subsystem for Linux (WSL) with PodMan installed. We will cover sandboxed execution in more detail later. 

In [ ]:
# | notest

executor = AgentExecutor(
     model='anthropic/claude-sonnet-4-5', # assumes ANTHROPIC_API_KEY is already set as environment variable
     sandbox=True,
     
 )

result = executor.run(
     task="""
     Create a simple Python calculator module with the following:
     - calculator.py with add, subtract, multiply, divide functions
     - test_calculator.py with pytest tests
     - All tests must pass
     """,
     working_dir='./calculator_project'
)
print(result)

time="2026-03-18T19:17:56-04:00" level=warning msg="The cgroupv2 manager is set to systemd but there is no systemd user session available"
time="2026-03-18T19:17:56-04:00" level=warning msg="For using systemd, you may need to login using an user session"
time="2026-03-18T19:17:56-04:00" level=warning msg="Alternatively, you can enable lingering with: `loginctl enable-linger 1000` (possibly as root)"
time="2026-03-18T19:17:56-04:00" level=warning msg="Falling back to --cgroup-manager=cgroupfs"

✈️  PatchPal Autopilot Mode Starting
Prompt: 
     Create a simple Python calculator module with the following:
     - calculator.py with add, su...
Completion promise: 'COMPLETE'
Max iterations: 50
Model: anthropic/claude-sonnet-4-5
Working directory: /workspace
🔒 File access restricted to working directory


🔄 Autopilot Iteration 1/50

🤔 Thinking...

I'll create a simple Python calculator module with tests. Let me start by       
creating the calculator.py file with the four basic operations.  

### Web Research Agent

This example is a web research agent that only has access to the following tools: `web_search`, `web_fetch`, `write_file`.

In [ ]:
# | notest

prompt = """
Research the latest developments in quantum computing in 2026.
Create a markdown report called 'quantum_computing_2026.md' with:
- Executive summary
- Key breakthroughs
- Major companies/institutions involved
- Potential applications
- Sources and references
"""

In [ ]:
# | notest

executor = AgentExecutor(
     model='anthropic/claude-sonnet-4-5',
     max_iterations=10,
    enabled_tools=["web_search", "web_fetch", "write_file"],
 )

result = executor.run(
     task=prompt,
     working_dir='./quantum_report'
)


✈️  PatchPal Autopilot Mode Starting
Prompt: 
Research the latest developments in quantum computing in 2026.
Create a markdown report called 'qua...
Completion promise: 'COMPLETE'
Max iterations: 10
Model: anthropic/claude-sonnet-4-5
Working directory: /home/amaiya/projects/ghub/onprem/nbs/quantum_report
🔒 File access restricted to working directory


🔄 Autopilot Iteration 1/10

🤔 Thinking...

I'll research the latest developments in quantum computing in 2026 and create a 
comprehensive markdown report.                                                  

🌐 Searching web: quantum computing breakthroughs 2026
🌐 Searching web: quantum computing companies developments 2026
🌐 Searching web: quantum computing applications 2026
🤔 Thinking...

Now let me fetch detailed information from some of the key sources:             

🌐 Fetching: https://www.forbes.com/sites/bernardmarr/2025/12/11/7-quantum-computing-trends-that-will-shape-every-industry-in-2026/
✗ web_fetch: Failed to fetch URL: 403 Cli

In [ ]:
!pwd

/home/amaiya/projects/ghub/onprem/nbs


In [ ]:
# | notest

from IPython.display import display, HTML
from markdown import markdown

with open('./quantum_report/quantum_computing_2026.md', 'r') as f:
    lines = [f.readline() for _ in range(20)]
    content = ''.join(lines)

html_content = markdown(content)

display(HTML(f"""
 <div style="
     border-left: 4px solid #4CAF50;
     padding: 15px 20px;
     margin: 15px 0;
     background-color: #f0f7f4;
     border-radius: 4px;
     box-shadow: 0 2px 4px rgba(0,0,0,0.1);
 ">
     <h4 style="margin-top: 0; color: #2e7d32;">📄 First 20 lines of markdown report from agent:</h4>
     {html_content}
 </div>
 """))

### Local Models

The `AgentExecutor` supports local models.  By default, it will assume the local model supports native function-calling (e.g., gpt-oss-120b).  If you use a local model that does **not** have good native support for function-calling (a.k.a. tool-calling), you can change the agent_type to `react`. In this example, we will use `glm-4.7-flash`.

**Note:** The default context window length in Ollama is typically too small for agentic  workflows.  Depending on the model and task, we recommend inreasing to at least 8192.  Reasoning models like gpt-oss:120b may require 32K or 64K.  

```sh
OLLAMA_CONTEXT_LENGTH=32000 ollama serve
```

In [ ]:
# | notest

executor = AgentExecutor(
     model='ollama_chat/glm-4.7-flash:q4_K_M',
     enabled_tools=["web_fetch", "write_file"],
     max_iterations=10
 )

result = executor.run(
     task="What is the highest level of education of the person listed on this page: https://arun.maiya.net? Write answer in answer.txt.",
     working_dir='./extraction_example'
)


✈️  PatchPal Autopilot Mode Starting
Prompt: What is the highest level of education of the person listed on this page: https://arun.maiya.net? Wr...
Completion promise: 'COMPLETE'
Max iterations: 10
Model: ollama_chat/glm-4.7-flash:q4_K_M
Working directory: /home/amaiya/projects/ghub/onprem/nbs/extraction_example
🔒 File access restricted to working directory


🔄 Autopilot Iteration 1/10

🤔 Thinking...

I'll fetch the webpage to find the highest level of education listed for the    
person.                                                                         

🌐 Fetching: https://arun.maiya.net
🤔 Thinking...

I found the education information on the page. The person states: "I completed a
Ph.D. in Computer Science at the Laboratory for Computational Population        
Biology..." The highest level of education is a Ph.D.                           

I'll write this to answer.txt.                                                  

📝 Patching: answer.txt
🤔 Thinking...

📝 Agent Response

In [ ]:
# | notest

!ls ./extraction_example/

answer.txt


In [ ]:
# | notest

!cat ./extraction_example/answer.txt

Ph.D. in Computer Science

### Sandboxed Execution

For enhanced security and isolation, set `sandbox=True` to run the agent in an ephemeral Docker/Podman container. This is useful when working with
untrusted code, needing resource limits, or wanting to protect your file system from accidental modifications. 

> Prerequisites: Requires Docker or Podman installed. See docker.com or podman.io.


In [ ]:
# | notest

prompt = """
Create a Python script that:
1. Generates sample sales data for 12 months (random)
2. Calculates total sales, average, min, max
3. Creates a matplotlib bar chart saved as 'sales_chart.png'
4. Writes a summary report to 'sales_analysis.txt'
"""

In [ ]:
# | notest

executor = AgentExecutor(
     model='anthropic/claude-sonnet-4-5',
     max_iterations=10,
    sandbox=True
 )

result = executor.run(
     task=prompt,
     working_dir='./data_analysis'
)

time="2026-03-18T19:21:56-04:00" level=warning msg="The cgroupv2 manager is set to systemd but there is no systemd user session available"
time="2026-03-18T19:21:56-04:00" level=warning msg="For using systemd, you may need to login using an user session"
time="2026-03-18T19:21:56-04:00" level=warning msg="Alternatively, you can enable lingering with: `loginctl enable-linger 1000` (possibly as root)"
time="2026-03-18T19:21:56-04:00" level=warning msg="Falling back to --cgroup-manager=cgroupfs"

✈️  PatchPal Autopilot Mode Starting
Prompt: 
Create a Python script that:
1. Generates sample sales data for 12 months (random)
2. Calculates to...
Completion promise: 'COMPLETE'
Max iterations: 10
Model: anthropic/claude-sonnet-4-5
Working directory: /workspace
🔒 File access restricted to working directory


🔄 Autopilot Iteration 1/10

🤔 Thinking...

I'll create a Python script that generates sample sales data, performs          
calculations, creates a visualization, and writes a summary repor

In [ ]:
# | notest
!ls ./data_analysis/

sales_analysis.py  sales_analysis.txt  sales_chart.png


In [ ]:
 # | notest
 from IPython.display import display, HTML

 with open('./data_analysis/sales_analysis.txt', 'r') as f:
     lines = [f.readline() for _ in range(50)]
     content = ''.join(lines)

 display(HTML(f"""
 <div style="
     border-left: 4px solid #4CAF50;
     padding: 15px 20px;
     margin: 15px 0;
     background-color: #f0f7f4;
     border-radius: 4px;
     box-shadow: 0 2px 4px rgba(0,0,0,0.1);
 ">
     <h4 style="margin-top: 0; color: #2e7d32;">📄 First 50 lines of sales report from agent:</h4>
     <pre style="white-space: pre-wrap; margin: 0;">{content}</pre>
 </div>
 """))

 ### Local Models + Sandbox: Networking Setup

 Local models (Ollama, llama.cpp) on localhost need container networking configured:

 - **Linux/WSL2**: Supply `network='host'` to `AgentExecutor`. (WSL2: make sure to enable mirrored networking in `.wslconfig`.)
 - **macOS/Windows**: Set `OLLAMA_API_BASE='http://host.docker.internal:11434'` (Docker) or `http://host.containers.internal:11434` (Podman)

### Custom Tools

You can give the agent custom tools by simply defining them as Python functions or callables.

In this example, we'll build a financial analysis agent with custom tools.

Let's first define the custom tools, which are based on yfinance. 

> `pip install yfinance`


#### Step 1: Define the  custom tools as Python functions

In [ ]:
# | notest

from typing import List, Dict
from datetime import datetime, timedelta

# Define custom financial tools


def get_current_stock_price(ticker: str) -> Dict[str, float]:
    """
    Fetch current/live stock price for a given ticker.
    
    Args:
        ticker: Stock ticker symbol (e.g., 'AAPL', 'GOOGL')
     Returns:
        Dictionary with current price and related info
    """
    try:
        import yfinance as yf
        from datetime import datetime
        stock = yf.Ticker(ticker)
        info = stock.info

        # Get current price (live during market hours, last close otherwise)
        current_price = info.get('currentPrice') or info.get('regularMarketPrice')

        return {
            "ticker": ticker.upper(),
            "current_price": round(current_price, 2),
            "market_state": info.get('marketState', 'unknown'),  # 'REGULAR', 'CLOSED', etc.
            "timestamp": datetime.now().isoformat()
        }
    except Exception as e:
        return {"error": str(e)}


def calculate_return_percentage(purchase_price: float, current_price: float) -> float:
     """
     Calculate percentage return on investment.
    
     Args:
         purchase_price: Original purchase price per share
         current_price: Current market price per share
    
     Returns:
         Percentage return (positive for gains, negative for losses)
     """
     if purchase_price == 0:
         return 0.0
     return round(((current_price - purchase_price) / purchase_price) * 100, 2)

def analyze_volatility(ticker: str, days: int = 30) -> Dict[str, float]:
     """
     Calculate stock volatility metrics over a period.
    
     Args:
         ticker: Stock ticker symbol
         days: Number of days to analyze
    
     Returns:
         Dictionary with volatility metrics
     """
     try:
         import yfinance as yf
         stock = yf.Ticker(ticker)
         end_date = datetime.now()
         start_date = end_date - timedelta(days=days + 10)
         hist = stock.history(start=start_date, end=end_date)
    
         if hist.empty or len(hist) < 2:
             return {"error": f"Insufficient data for {ticker}"}
    
         daily_changes = hist['Close'].pct_change() * 100
    
         return {
             "ticker": ticker.upper(),
             "period_days": len(hist),
             "avg_daily_change": round(abs(daily_changes.mean()), 2),
             "max_increase": round(daily_changes.max(), 2),
             "max_decrease": round(daily_changes.min(), 2),
             "std_deviation": round(daily_changes.std(), 2)
         }
     except Exception as e:
         return {"error": str(e)}

#### Step 2: Launch the agent within with access to the custom tools

In [ ]:
# | notest

# Create agent with custom tools
from onprem.pipelines.agent import AgentExecutor

executor = AgentExecutor(
 model='anthropic/claude-sonnet-4-5',
 custom_tools=[calculate_return_percentage, analyze_volatility, get_current_stock_price],
 enabled_tools=['write_file'],
 verbose=True
)

# Task: Analyze Apple and Microsoft stock
task = """
Create a stock analysis report for Apple (AAPL) and Microsoft (MSFT):

1. Get current stock prices for both companies
2. Analyze volatility for both over the last 30 days
3. If I bought AAPL at $150 and it's now at current price, calculate my return percentage
4. Create a markdown report comparing the two stocks

Save the report to 'stock_analysis.md'
"""

result = executor.run(task, working_dir='./financial_workspace')

✓ Wrote custom tool 'calculate_return_percentage' to .patchpal/tools/calculate_return_percentage.py
✓ Wrote custom tool 'analyze_volatility' to .patchpal/tools/analyze_volatility.py
✓ Wrote custom tool 'get_current_stock_price' to .patchpal/tools/get_current_stock_price.py

✈️  PatchPal Autopilot Mode Starting
Prompt: 
Create a stock analysis report for Apple (AAPL) and Microsoft (MSFT):

1. Get current stock prices ...
Completion promise: 'COMPLETE'
Max iterations: 50
Model: anthropic/claude-sonnet-4-5
Working directory: /home/amaiya/projects/ghub/onprem/nbs/financial_workspace
🔒 File access restricted to working directory
🔧 Custom tools: analyze_volatility, calculate_return_percentage, get_current_stock_price


🔄 Autopilot Iteration 1/50

🤔 Thinking...

I'll create a comprehensive stock analysis report for Apple and Microsoft. Let  
me gather the data first.                                                       

🔧 get_current_stock_price({'ticker': 'AAPL'})
🔧 get_current_stock_price

In [ ]:
# | notest

from IPython.display import display, HTML
from markdown import markdown

with open('./financial_workspace/stock_analysis.md', 'r') as f:
    lines = [f.readline() for _ in range(30)]
    content = ''.join(lines)

html_content = markdown(content, extensions=['tables'])

display(HTML(f"""
 <div style="
     border-left: 4px solid #4CAF50;
     padding: 15px 20px;
     margin: 15px 0;
     background-color: #f0f7f4;
     border-radius: 4px;
     box-shadow: 0 2px 4px rgba(0,0,0,0.1);
 ">
     <h4 style="margin-top: 0; color: #2e7d32;">📄 First 30 lines of markdown report from agent:</h4>
     {html_content}
 </div>
 """))

Ticker,Company,Current Price
AAPL,Apple Inc.,$254.46
MSFT,Microsoft Corporation,$399.14
